In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
# hyperparams

batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 200
learning_rate = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 50
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2

torch.manual_seed(1337)

In [ ]:
with open('/content/drive/MyDrive/projects_neural_practice/input.txt', 'r', encoding = 'utf-8') as f:
  text = f.read()

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

# create the mapping
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l : ''.join(itos[i] for i in l)

In [ ]:
# Train and test splits

data = torch.tensor(encode(text), dtype = torch.long)
n = int(0.9 * len(data)) # 90% train
train_data = data[:n]
val_data = data[n:]

In [ ]:
# dataloader

def get_batch(split):

  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

In [ ]:
@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

In [ ]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  #def forward(self, x):
  #  B, T, C = x.shape
  #  k = self.key(x)
  #  q = self.query(x)

    # compute attention scores
  #  wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
  #  wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
  #  wei = F.softmax(wei, dim = -1)
  #  wei = self.dropout(wei) # from the code of Karpathy

  #  v = self.value(x)
  #  out = wei @ v
  #  return out

  def forward(self, x):
    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x)
    v = self.value(x)

    out = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=None,
        dropout_p=dropout if self.training else 0.0,
        is_causal=True,
    )

    return out

In [ ]:
class MultiHeadAttention(nn.Module):

  def __init__(self, num_heads, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(head_size * num_heads, n_embd)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim = -1)
    out = self.dropout(self.proj(out))
    return out

In [ ]:
class FeedForward(nn.Module):

  def __init__(self, n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd, n_embd*4),
        nn.ReLU(),
        nn.Linear(n_embd*4, n_embd),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)

In [ ]:
class Block(nn.Module):

  def __init__(self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

In [ ]:
class GPTLanguageModel(nn.Module):

  def __init__(self):
    super().__init__()

    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head = n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final LN
    self.lm_head = nn.Linear(n_embd, vocab_size)
    self.apply(self._init_weights)

  def _init_weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean = 0.0, std = 0.02)

  def forward(self, idx, targets = None):
    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx) # (B, T, C)
    pos_emb = self.position_embedding_table(torch.arange(T, device = device)) # (T, C)
    x = tok_emb + pos_emb # (B, T, C)
    x = self.blocks(x) # (B, T, C)
    x = self.ln_f(x) # (B, T, C)
    logits = self.lm_head(x) # (B, T, vocab_size)

    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]
      logits, loss = self(idx_cond)
      logits = logits[:, -1, :]
      probs = F.softmax(logits, dim = -1)
      idx_next = torch.multinomial(probs, num_samples = 1)
      idx = torch.cat((idx, idx_next), dim = 1)
    return idx

In [ ]:
model = GPTLanguageModel().to(device)
model = torch.compile(model)
#m = model.to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, fused = True)

# FP16 gradient scaler
scaler = torch.amp.GradScaler("cuda")

import time
step_times = []

for iter in range(max_iters):

  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    avg_time = sum(step_times) / len(step_times) if step_times else 0
    tokens_per_step = batch_size * block_size
    tokens_per_sec = tokens_per_step / avg_time if avg_time > 0 else 0

    print(
        f"step {iter}: "
        f"train loss {losses['train']:.4f}, "
        f"val loss {losses['val']:.4f}, "
        f"avg step time {avg_time * 1000:.2f} ms, "
        f"tokens/step {tokens_per_step:,}, "
        f"tokens/sec {tokens_per_sec:,.0f}"
    )

    step_time = []

  xb, yb = get_batch('train')

  if device == "cuda":
    torch.cuda.synchronize()
  start = time.perf_counter()

  optimizer.zero_grad(set_to_none = True)
  # FP16 mixed precision forward pass
  with torch.autocast(device_type="cuda", dtype=torch.float16):
      logits, loss = model(xb, yb)

  # scaled backward pass
  scaler.scale(loss).backward()

  # optimizer step
  scaler.step(optimizer)
  scaler.update()


  if device == "cuda":
    torch.cuda.synchronize()

  step_times.append(time.perf_counter() - start)
  #elapsed = time.perf_counter() - start

  #print(f"Step {iter}: {elapsed:.4f} seconds")


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


10.788929 M parameters


W0917 19:19:26.379000 30306 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


step 0: train loss 4.2213, val loss 4.2304, avg step time 0.00 ms, tokens/step 16,384, tokens/sec 0
step 200: train loss 2.3962, val loss 2.4373, avg step time 465.98 ms, tokens/step 16,384, tokens/sec 35,160
step 400: train loss 1.9250, val loss 2.0335, avg step time 308.18 ms, tokens/step 16,384, tokens/sec 53,163
step 600: train loss 1.6222, val loss 1.8032, avg step time 256.22 ms, tokens/step 16,384, tokens/sec 63,946
step 800: train loss 1.4739, val loss 1.6756, avg step time 229.77 ms, tokens/step 16,384, tokens/sec 71,307
step 1000: train loss 1.3961, val loss 1.6143, avg step time 214.07 ms, tokens/step 16,384, tokens/sec 76,536
step 1200: train loss 1.3277, val loss 1.5699, avg step time 203.50 ms, tokens/step 16,384, tokens/sec 80,512
step 1400: train loss 1.2859, val loss 1.5480, avg step time 195.91 ms, tokens/step 16,384, tokens/sec 83,632
step 1600: train loss 1.2539, val loss 1.5237, avg step time 190.20 ms, tokens/step 16,384, tokens/sec 86,139
step 1800: train loss 1.